<a href="https://colab.research.google.com/github/f247805/thesis/blob/main/distilbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers
!pip install transformers datasets torch scikit-learn


Found existing installation: transformers 4.49.0
Uninstalling transformers-4.49.0:
  Successfully uninstalled transformers-4.49.0
  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
Using cached transformers-4.49.0-py3-none-any.whl (10.0 MB)


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


In [3]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Upload dataset
from google.colab import files
uploaded = files.upload()


import pandas as pd
df = pd.read_json("FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl", lines=True)
df.head()

# Preprocessing: Ensure correct column names
TEXT_COLUMN = "text"
ASPECT_COLUMN = "aspect"
LABEL_COLUMN = "sentiment"  # Adjust based on your dataset



Saving FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl to FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl


In [ ]:
print(df.head())

In [4]:
# Function to expand rows so that each (text, aspect) pair is a separate row
def expand_aspects(df):
    new_rows = []
    for _, row in df.iterrows():
        text = row["text"]
        aspects = row["labels"]  # This contains a list of {'aspect': ..., 'polarity': ...}

        for aspect_info in aspects:
            new_rows.append({
                "text": text,
                "aspect": aspect_info["aspect"],
                "label": aspect_info["polarity"]
            })

    return pd.DataFrame(new_rows)

In [5]:
# Expand dataset
df_expanded = expand_aspects(df)

In [6]:
# Convert labels to numerical values
label_map = {"positive": 2, "neutral": 1, "negative": 0}
df_expanded["label"] = df_expanded["label"].map(label_map)

# Format input text as: "text [SEP] aspect"
df_expanded["input_text"] = df_expanded["text"] + " [SEP] " + df_expanded["aspect"]

# Split into train & validation
train_data, val_data = train_test_split(df_expanded, test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)


In [7]:
#Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [8]:
# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["input_text"], padding="max_length", truncation=True)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns
train_dataset = train_dataset.remove_columns(["text", "aspect", "input_text"])
val_dataset = val_dataset.remove_columns(["text", "aspect", "input_text"])

print(train_dataset)

Map:   0%|          | 0/24650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6163 [00:00<?, ? examples/s]

Dataset({
    features: ['label', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 24650
})


In [9]:
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
import torch

# Define number of labels (positive, neutral, negative)
num_labels = 3

# Load model with classification head
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    report_to="none"  # This disables W&B logging
)


In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)


<ipython-input-11-62670ca281bb>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"


In [13]:
trainer.train()


Step,Training Loss
500,0.730200
1000,0.560600
1500,0.538200
2000,0.504600
2500,0.507900
3000,0.464400
3500,0.357900
4000,0.327500
4500,0.326800
5000,0.337300


TrainOutput(global_step=9246, training_loss=0.3608730455613513, metrics={'train_runtime': 3804.123, 'train_samples_per_second': 19.439, 'train_steps_per_second': 2.431, 'total_flos': 9796138827724800.0, 'train_loss': 0.3608730455613513, 'epoch': 3.0})

In [ ]:
trainer.evaluate()
